this notebook is for deciding on some model parameters and module development. namely, we will have to decide:

1. the embedding dimension, and positional embedding
2. the number of heads for multi-head attention, and MLP dimension
3. training schedule and parameters

in the spirit of how our other parameters have been chosen, we use GPT-2 as a benchmark for our model parameters. GPT-2 had variable embedding dimensions depending on the model size, with 768 dimensions for the smallest model. we choose the same dimensions to avoid the compute cost of figuring out the optimal embedding size. we will revisit the optimality of our decisions at a later point.

on the point of the amount of heads, this is generally backed out of each head's output dimension. a common choice is 64 (which is also the ), which would suggest 12 heads for our embedding dimension of 768; but empirical research points to the fact that the number of heads often matters much less than one would assume (Michel & Levy, 2019); so we will not ponder too much on this problem.

in terms of training schedules and parameters, this is the main empirical addition to our research in this notebook. but first, we must write modules which will allow us to train our neural network. of course, the below declarations are purely for development and documentation purposes, and standalone modules are contained in some appropiate path in the project.

In [13]:
import torch
import numpy as np
import torch.nn as nn

class MultiHeadAttentionDecoder(nn.Module):
    def __init__(self, n_dim: int, n_heads: int):
        super().__init__()
        if n_dim <= 0 or n_heads <= 0:
            raise ValueError('MultiHeadAttention: one or more parameters passed in is negative.')
        elif n_dim % n_heads != 0:
            raise ValueError('MultiHeadAttention: number of heads is not divisible by the embedding dimension')

        # linear projection layers to split later
        self.attn_proj = nn.Linear(n_dim, n_dim * 3)
        self.n_heads = n_heads

        # post-concat mixing of multi-head attention outputs
        self.out_proj = nn.Linear(n_dim, n_dim)

    def forward(self, x: torch.Tensor):
        x_proj: torch.Tensor = self.attn_proj(x)
        q, k, v = x_proj.chunk(3, dim=-1)

        # batches, tokens and embedding dimension
        B, T, C = q.shape
        H = self.n_heads
        D = C // H

        # split the output dimension into heads; and then transpose such that heads is the second dimension
        # this means that matrix multiplications will treat the (batch, head) as the batch dimension
        q = q.view(B, T, H, D).transpose(1, 2)
        k = k.view(B, T, H, D).transpose(1, 2)
        v = v.view(B, T, H, D).transpose(1, 2)

        # get attention scores
        attention = q @ k.transpose(-2, -1)
        attention = attention / np.sqrt(D)

        mask = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1)

        # causal mask, then soft max on keys
        attention = attention.masked_fill(mask, float('-inf'))
        attention = attention.softmax(dim=-1)

        out = attention @ v
        # remerge output, then pass through linear output layer
        out = out.transpose(1, 2).reshape(B, T, C)

        return self.out_proj(out)

we can produce a small example of the scores that the attention block should produce.

In [14]:
x = torch.tensor([
    [[ 0.1, -0.2,  0.3,  0.0,  0.5, -0.1,  0.2,  0.4],
     [ 0.7,  0.1, -0.3,  0.2,  0.0,  0.6, -0.5,  0.1],
     [-0.4,  0.3,  0.8, -0.2,  0.1,  0.0,  0.3, -0.6],
     [ 0.2,  0.9, -0.1,  0.5, -0.3,  0.2,  0.0,  0.7]]
])

mtah = MultiHeadAttentionDecoder(n_dim=8, n_heads=2)
out = mtah(x)
print(out)

tensor([[[-0.1735,  0.3221, -0.2205,  0.2879, -0.5726, -0.4255,  0.1790,
          -0.1777],
         [-0.0107,  0.3591, -0.2606,  0.2530, -0.5052, -0.4177,  0.1176,
          -0.1998],
         [-0.1065,  0.3538, -0.2394,  0.2164, -0.4393, -0.3714,  0.1272,
          -0.1672],
         [-0.0346,  0.3853, -0.2237,  0.2137, -0.4099, -0.3325,  0.0847,
          -0.1111]]], grad_fn=<UnsafeViewBackward0>)


each block in a transformer contains an attention block and a feed forward/MLP block, so we can collate the two blocks into one module; i repeat imports so that these code cells are individual modules.

In [15]:
import torch
import torch.nn as nn

class TransformerBlock(nn.Module):
    def __init__(self, n_dim: int, n_heads: int, mlp_dim: int):
        super().__init__()
        if n_dim <= 0 or n_heads <= 0: raise ValueError('MultiHeadAttention: one or more parameters passed in is negative.')
        elif n_dim % n_heads != 0: raise ValueError('MultiHeadAttention: number of heads is not divisible by the embedding dimension')

        self.attn = MultiHeadAttentionDecoder(n_dim, n_heads)
        self.ln1 = nn.LayerNorm(n_dim)
        self.mlp = nn.Sequential(nn.Linear(n_dim, mlp_dim), nn.GELU(), nn.Linear(mlp_dim, n_dim))
        self.ln2 = nn.LayerNorm(n_dim)

    def forward(self, x: torch.Tensor):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

In [32]:
tb = TransformerBlock(n_dim=8, n_heads=2, mlp_dim=8*4)
out = tb(x)
print(out)

tensor([[[ 0.7802, -0.2275, -0.0766, -0.0086,  0.8606, -0.4433,  0.0483,
           0.3213],
         [ 1.1356, -0.4626, -0.7462,  0.4805,  0.4162,  0.7873, -0.4151,
          -0.1519],
         [-0.0519, -0.0711,  0.8044, -0.5389, -0.0704, -0.4538,  0.7937,
          -0.3644],
         [ 0.7708,  0.7018, -0.0128,  0.4007, -0.2740, -0.0305,  0.1102,
           0.6581]]], grad_fn=<AddBackward0>)


now to create a full transformer neural net, we generally stack many of these transformer blocks in sequence. each transformer block is a self contained, residual connected block, so we can pass through the outputs of each block into another without any processing.

In [17]:
import torch
import torch.nn as nn
from typing import List

class TransformerNet(nn.Module):
    def __init__(self, n_dim: int, n_heads: List[int], mlp_dim: List[int]):
        super().__init__()
        if len(n_heads) != len(mlp_dim): raise ValueError('TransformerNet: the number of values given in n_heads does not equal the number of values in mlp_dim.')
        self.blocks = [ TransformerBlock(n_dim, n_heads[i], mlp_dim[i]) for i in range(len(n_heads)) ]

    def forward(self, x: torch.Tensor):
        for block in self.blocks: x = block(x)
        return x

In [36]:
tn = TransformerNet(n_dim=8, n_heads=[2, 2, 2, 2], mlp_dim=[8*4, 8*4, 8*4, 8*4])
out = tn(x)
print(out)

tensor([[[-0.5344, -2.2727, -1.0084,  0.8444, -0.2542, -0.8253,  0.0849,
           0.2100],
         [ 1.3553, -1.1297, -1.3391,  1.2216, -0.2493, -0.2291, -1.0583,
           0.2464],
         [-0.5957, -0.3674, -0.1381,  0.0470, -1.1984, -0.9476, -0.1381,
          -0.2916],
         [ 0.1745,  0.4654, -1.2327,  1.1192, -1.2878, -0.6584, -0.2931,
           1.6706]]], grad_fn=<AddBackward0>)
